# ML-06 — Signal Audit: Do Pair-Interaction Signals Hold?

Verdicts are CONFIRMED, OPPOSITE, MIXED, or FALSE for this development cohort only.

## 1. Distributions

Overlap and demand are heavy-tailed; use quantiles, logs, and evidence floors rather than means.

In [1]:
from pathlib import Path
import duckdb, numpy as np, pandas as pd
def find_root(p=Path.cwd()):
    for x in [p,*p.parents]:
        if (x/"skills"/"README.md").exists(): return x
    raise FileNotFoundError("Run inside repository")
ROOT=find_root()
PAIR_PATH=ROOT/"work"/"outputs"/"page_pair_features.parquet"
assert PAIR_PATH.exists() and PAIR_PATH.stat().st_size>0, "Run work/scripts/build_pair_features.py"
con=duckdb.connect()
R=f"read_parquet('{PAIR_PATH.as_posix()}')"
con.sql(f"""SELECT COUNT(*) pairs,
QUANTILE_CONT(shared_query_count,[.1,.5,.9,.99]) shared_query_q,
QUANTILE_CONT(weighted_query_overlap,[.1,.5,.9,.99]) overlap_q,
QUANTILE_CONT(smaller_page_demand_overlap,[.1,.5,.9,.99]) demand_q,
QUANTILE_CONT(mean_shared_position_gap,[.1,.5,.9,.99]) position_gap_q FROM {R}""").df()

,pairs,shared_query_q,overlap_q,demand_q,position_gap_q
0,362562,"[2.0, 3.0, 15.0, 57.0]","[0.0019399084018091692, 0.01694697650532803, 0...","[0.005991355330301943, 0.04246457942558954, 0....","[2.078350503364955, 10.396159298149655, 33.699..."


## 2. Signal test 1 — overlap and position proximity

Hypothesis: stronger weighted overlap generally accompanies smaller shared-query position gaps.
This is supportive structure, not proof of competition.

In [2]:
from pathlib import Path
import duckdb, numpy as np, pandas as pd
def find_root(p=Path.cwd()):
    for x in [p,*p.parents]:
        if (x/"skills"/"README.md").exists(): return x
    raise FileNotFoundError("Run inside repository")
ROOT=find_root()
PAIR_PATH=ROOT/"work"/"outputs"/"page_pair_features.parquet"
assert PAIR_PATH.exists() and PAIR_PATH.stat().st_size>0, "Run work/scripts/build_pair_features.py"
con=duckdb.connect()
R=f"read_parquet('{PAIR_PATH.as_posix()}')"
t1=con.sql(f"""WITH q AS (SELECT *,NTILE(5) OVER
(ORDER BY weighted_query_overlap) oq FROM {R})
SELECT oq,COUNT(*) pairs,MEDIAN(weighted_query_overlap) median_overlap,
MEDIAN(mean_shared_position_gap) median_position_gap FROM q GROUP BY 1 ORDER BY 1""").df()
print("Verdict:","CONFIRMED" if t1.iloc[-1].median_position_gap<t1.iloc[0].median_position_gap else "OPPOSITE")
t1

Verdict: OPPOSITE


,oq,pairs,median_overlap,median_position_gap
0,1,72513,0.001940,9.721465
1,2,72513,0.007086,10.970235
2,3,72512,0.016947,10.807925
3,4,72512,0.037736,10.290719
4,5,72512,0.103646,10.189067


## 3. Signal test 2 — movement is not one-dimensional

Hypothesis: shared-growth, shared-decline, and opposite-movement regimes are all substantial.
Substitution is therefore a feature, not a candidate gate.

In [3]:
from pathlib import Path
import duckdb, numpy as np, pandas as pd
def find_root(p=Path.cwd()):
    for x in [p,*p.parents]:
        if (x/"skills"/"README.md").exists(): return x
    raise FileNotFoundError("Run inside repository")
ROOT=find_root()
PAIR_PATH=ROOT/"work"/"outputs"/"page_pair_features.parquet"
assert PAIR_PATH.exists() and PAIR_PATH.stat().st_size>0, "Run work/scripts/build_pair_features.py"
con=duckdb.connect()
R=f"read_parquet('{PAIR_PATH.as_posix()}')"
t2=con.sql(f"""SELECT CASE
WHEN growth_a IS NULL OR growth_b IS NULL THEN 'insufficient_history'
WHEN growth_a>0 AND growth_b>0 THEN 'shared_growth'
WHEN growth_a<0 AND growth_b<0 THEN 'shared_decline'
WHEN growth_a*growth_b<0 THEN 'opposite_movement' ELSE 'flat_or_mixed' END pattern,
COUNT(*) pairs,MEDIAN(weighted_query_overlap) median_overlap,
MEDIAN(visibility_balance) median_balance FROM {R} GROUP BY 1 ORDER BY pairs DESC""").df()
needed={"shared_growth","shared_decline","opposite_movement"}
print("Verdict:","CONFIRMED" if needed<=set(t2.loc[t2.pairs>=1000,"pattern"]) else "FALSE")
t2

Verdict:

 CONFIRMED


,pattern,pairs,median_overlap,median_balance
0,shared_decline,178522,0.015546,0.364012
1,opposite_movement,122737,0.017142,0.351772
2,shared_growth,45251,0.019812,0.410540
3,insufficient_history,12582,0.026361,0.285621
4,flat_or_mixed,3470,0.026670,0.349547


## 4. Signal test 3 — balanced fragmentation differs from dominance

Hypothesis: evidence-rich overlap contains both balanced and dominant relationships, supporting
different editorial reviews.

In [4]:
from pathlib import Path
import duckdb, numpy as np, pandas as pd
def find_root(p=Path.cwd()):
    for x in [p,*p.parents]:
        if (x/"skills"/"README.md").exists(): return x
    raise FileNotFoundError("Run inside repository")
ROOT=find_root()
PAIR_PATH=ROOT/"work"/"outputs"/"page_pair_features.parquet"
assert PAIR_PATH.exists() and PAIR_PATH.stat().st_size>0, "Run work/scripts/build_pair_features.py"
con=duckdb.connect()
R=f"read_parquet('{PAIR_PATH.as_posix()}')"
t3=con.sql(f"""WITH h AS (SELECT * FROM {R}
WHERE weighted_query_overlap>=.10 AND smaller_page_demand_overlap>=.20)
SELECT CASE WHEN visibility_balance>=.67 THEN 'balanced_fragmentation'
WHEN visibility_balance<=.33 THEN 'dominant_sibling' ELSE 'moderately_imbalanced' END relationship,
COUNT(*) pairs,MEDIAN(mean_shared_position_gap) median_position_gap,
MEDIAN(shared_impression_intersection) median_shared_demand FROM h GROUP BY 1 ORDER BY pairs DESC""").df()
print("Verdict:","CONFIRMED" if {"balanced_fragmentation","dominant_sibling"}<=set(t3.relationship) else "FALSE")
t3

Verdict: CONFIRMED


,relationship,pairs,median_position_gap,median_shared_demand
0,moderately_imbalanced,11648,10.330953,178.0
1,balanced_fragmentation,6597,9.927074,188.0
2,dominant_sibling,6110,11.814050,174.0


## 5. Underlying flag-assumption test

Product flags are absent. Test the underlying assumption: raw overlap is too broad, so an
actionable review needs material overlap and position evidence.

In [5]:
from pathlib import Path
import duckdb, numpy as np, pandas as pd
def find_root(p=Path.cwd()):
    for x in [p,*p.parents]:
        if (x/"skills"/"README.md").exists(): return x
    raise FileNotFoundError("Run inside repository")
ROOT=find_root()
PAIR_PATH=ROOT/"work"/"outputs"/"page_pair_features.parquet"
assert PAIR_PATH.exists() and PAIR_PATH.stat().st_size>0, "Run work/scripts/build_pair_features.py"
con=duckdb.connect()
R=f"read_parquet('{PAIR_PATH.as_posix()}')"
ft=con.sql(f"""SELECT COUNT(*) all_pairs,
SUM((weighted_query_overlap>=.10 AND smaller_page_demand_overlap>=.20
AND mean_shared_position_gap<=20)::INT) evidence_rich_pairs FROM {R}""").df()
ft["evidence_rich_share"]=ft.evidence_rich_pairs/ft.all_pairs
print("Verdict:","CONFIRMED" if ft.evidence_rich_share.iloc[0]<.5 else "MIXED")
ft

Verdict: CONFIRMED


,all_pairs,evidence_rich_pairs,evidence_rich_share
0,362562,20024.0,0.055229


## 6. Practical meaning

The audit supports a broad interaction model, not an opposite-movement detector. Overlap,
proximity, balance, and movement add different information. Shared growth is not automatically
bad; shared decline may be topic weakness; pruning remains human review because page purpose and
conversion value are unavailable.